# 

# Variables de Contexto Socioeconómico por Entidad Federativa

## Importación de Librerías y Configuración

In [173]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import geopandas as gpd
import unicodedata

RAW_DIR = Path("../data/raw")
STATE_MASTER_FILE = Path("../data/catalogs/states.master.json")
PIB_FILE = RAW_DIR / "PIBE_2.xlsx"
POBLACION_FILE = RAW_DIR / "conjunto_de_datos_iter_00CSV20.csv"
SHAPEFILE_PATH = RAW_DIR / "00ent.shp"
CONEVAL_FILE = RAW_DIR / "Anexo estadístico entidades 2022.xlsx"

def clean_entity_code(x):
    return str(x).zfill(2)

def normalize(text):
    if text is None:
        return ""
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    return text


def build_state_lookup(state_master):
    lookup = {}

    for cve, data in state_master.items():
        names = [data["estado"]] + data.get("aliases", [])

        for name in names:
            lookup[normalize(name)] = cve

    return lookup

## PIB - PIBE 2024

In [174]:
def load_pib():
    df = pd.read_excel(PIB_FILE, skiprows=4)

    # columna "concepto" tiene valores únicos, es la que usaremos para mergear con coneval
    df = df.iloc[3:35]
    last_year = df.columns[-1]

    df = df[["Concepto", last_year]].rename(columns={"Concepto": "estado", last_year: "pib_total"})
    return df

## Datos de Población - ITER 2020

In [175]:
def load_poblacion():
    # ITER - solo nivel estatal
    try:
        df = pd.read_csv(
            POBLACION_FILE,
            dtype={"MUN": int, "LOC": int, "ENTIDAD": int, "POBTOT": int},
            usecols=["MUN", "LOC", "ENTIDAD", "POBTOT", "POB15_64", "PEA", "PDER_IMSS", "VIVTOT"]
        )
    except FileNotFoundError:
        raise FileNotFoundError("El archivo conjunto_de_datos_iter_00CSV20.csv no se encuentra en la ruta especificada.")
    except ValueError as e:
        raise ValueError(f"Error al cargar el archivo: {e}")

    df = df[
        (df["MUN"] == 0) &
        (df["LOC"] == 0) &
        (df["ENTIDAD"] != 0)
    ]

    # Renombrar columnas relevantes
    df = df.rename(columns={
        "ENTIDAD": "cve_ent",
        "POBTOT": "poblacion_total",
        "POB15_64": "poblacion_edad_laboral",
        "PEA": "poblacion_economicamente_activa",
        "PDER_IMSS": "poblacion_afiliada_imss",
        "VIVTOT": "total_viviendas"
    })

    # Asegurar que los códigos de entidad estén rellenados con ceros
    df["cve_ent"] = df["cve_ent"].apply(clean_entity_code)

    # Seleccionar columnas relevantes
    return df[[
        "cve_ent", 
        "poblacion_total", 
        "poblacion_edad_laboral", 
        "poblacion_economicamente_activa", 
        "poblacion_afiliada_imss", 
        "total_viviendas"
    ]]

## Superficie

In [176]:
def load_surface_area():
    gdf = gpd.read_file(SHAPEFILE_PATH)

    gdf = gdf.to_crs(epsg=6372)  # Mexico ITRF2008 / LCC

    gdf["superficie_km2"] = gdf.geometry.area / 1e6
    df_geo = gdf[["CVE_ENT", "superficie_km2"]]

    return df_geo

## Densidad de Población

In [177]:
def compute_densidad(df_pob, df_geo):
    # Merge population data with geographic data
    df = df_pob.merge(df_geo, on="cve_ent")
    df["densidad_poblacional"] = df["poblacion_total"] / df["superficie_km2"]

    if "POB15_64" in df_pob.columns:
        df["poblacion_edad_laboral_pct"] = (df["POB15_64"] / df["poblacion_total"]) * 100

    return df[["cve_ent", "densidad_poblacional", "poblacion_edad_laboral_pct"]]

## CONEVAL 2022

In [178]:
def load_coneval():
    xls = pd.ExcelFile(CONEVAL_FILE)

    records = []

    for sheet in xls.sheet_names:
        if sheet == "Contenido":
            continue

        df = xls.parse(sheet, header=6)

        df.columns = df.columns.map(str)

        def get_value(label):
            row = df[df.iloc[:, 2].str.contains(label, na=False, case=False)]
            if row.empty:
                return None
            try:
                return float(row.iloc[0, 6])
            except (IndexError, ValueError):
                return None

        records.append({
            "estado": sheet,
            "pobreza_pct": get_value("Población en situación de pobreza"),
            "pobreza_extrema_pct": get_value("pobreza extrema"),
            "vulnerable_carencias_pct": get_value("vulnerable por carencias"),
            "no_pobre_no_vulnerable_pct": get_value("no pobre y no vulnerable"),
            "rezago_educativo_pct": get_value("Rezago educativo"),
            "carencia_salud_pct": get_value("salud"),
            "carencia_servicios_basicos_pct": get_value("servicios básicos"),
            "ingreso_inferior_lp_pct": get_value("línea de pobreza por ingresos"),
        })

    return pd.DataFrame(records)[1:]

In [179]:
def load_state_master() -> dict[str, dict]:
    payload = json.loads(STATE_MASTER_FILE.read_text(encoding="utf-8"))
    return {state["cve_ent"]: state for state in payload["states"]}

In [180]:
def build_dataset():
    pib = load_pib()
    pob = load_poblacion()
    coneval = load_coneval()

    # concatenar / merge
    state_master = load_state_master()
    lookup = build_state_lookup(state_master)

    coneval["cve_ent"] = coneval["estado"].apply(lambda x: lookup.get(normalize(x)))
    coneval.drop(columns=["estado"], inplace=True)

    pib["cve_ent"] = pib["estado"].apply(lambda x: lookup.get(normalize(x)))

    df = coneval.merge(pib, on="cve_ent", how="left").merge(pob, on="cve_ent", how="left")

    numeric_columns = [
        "pib_total", "poblacion_total", "poblacion_edad_laboral", 
        "poblacion_economicamente_activa", "poblacion_afiliada_imss", "total_viviendas"
    ]
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["pib_per_capita"] = df["pib_total"] / df["poblacion_total"]

    # Añadir columnas en porcentaje
    df["poblacion_edad_laboral_pct"] = (df["poblacion_edad_laboral"] / df["poblacion_total"]) * 100
    df["poblacion_economicamente_activa_pct"] = (df["poblacion_economicamente_activa"] / df["poblacion_total"]) * 100
    df["poblacion_afiliada_imss_pct"] = (df["poblacion_afiliada_imss"] / df["poblacion_total"]) * 100
    df["total_viviendas_pct"] = (df["total_viviendas"] / df["poblacion_total"]) * 100

    # Reordenar
    columns = ["cve_ent", "estado"] + [col for col in df.columns if col not in ["cve_ent", "estado"]]
    df = df[columns]

    return df

In [181]:
df_base = build_dataset()

## Exportación a formatos wide y long

In [182]:
SOURCE = "CONEVAL 2022 / INEGI ITER 2020 / INEGI PIB"
UPDATED_AT = "2026-04-22"
PROCESSED_DIR = Path("../data/processed")
PUBLIC_DIR = Path("../public/data")

VARIABLE_SPECS = [
    {"variable": "pobreza_pct",                       "anio": 2022, "categoria": "bienestar_social", "label": "Población en situación de pobreza",       "unidad": "%"},
    {"variable": "pobreza_extrema_pct",                "anio": 2022, "categoria": "bienestar_social", "label": "Población en pobreza extrema",            "unidad": "%"},
    {"variable": "vulnerable_carencias_pct",           "anio": 2022, "categoria": "bienestar_social", "label": "Población vulnerable por carencias",      "unidad": "%"},
    {"variable": "no_pobre_no_vulnerable_pct",         "anio": 2022, "categoria": "bienestar_social", "label": "Población no pobre y no vulnerable",      "unidad": "%"},
    {"variable": "rezago_educativo_pct",               "anio": 2022, "categoria": "bienestar_social", "label": "Rezago educativo",                        "unidad": "%"},
    {"variable": "carencia_salud_pct",                 "anio": 2022, "categoria": "bienestar_social", "label": "Carencia por acceso a salud",             "unidad": "%"},
    {"variable": "carencia_servicios_basicos_pct",     "anio": 2022, "categoria": "bienestar_social", "label": "Carencia por servicios básicos",          "unidad": "%"},
    {"variable": "ingreso_inferior_lp_pct",            "anio": 2022, "categoria": "bienestar_social", "label": "Ingreso inferior a la línea de pobreza", "unidad": "%"},
    {"variable": "pib_total",                          "anio": 2024, "categoria": "economia",         "label": "PIB total",                               "unidad": "MDP"},
    {"variable": "pib_per_capita",                     "anio": 2024, "categoria": "economia",         "label": "PIB per cápita",                          "unidad": "MDP"},
    {"variable": "poblacion_total",                    "anio": 2020, "categoria": "demografia",       "label": "Población total",                         "unidad": "personas"},
    {"variable": "poblacion_edad_laboral_pct",         "anio": 2020, "categoria": "demografia",       "label": "Población en edad laboral",               "unidad": "%"},
    {"variable": "poblacion_economicamente_activa_pct","anio": 2020, "categoria": "demografia",       "label": "Población económicamente activa",         "unidad": "%"},
    {"variable": "poblacion_afiliada_imss_pct",        "anio": 2020, "categoria": "demografia",       "label": "Población afiliada al IMSS",              "unidad": "%"},
]


In [183]:
def build_outputs(df):
    state_master = load_state_master()

    long_records = []
    metric_catalog = []

    for spec in VARIABLE_SPECS:
        metric_catalog.append({
            "variable_id": spec["variable"],
            "anio": spec["anio"],
            "categoria_id": spec["categoria"],
            "label": spec["label"],
            "unidad": spec["unidad"],
        })

        for _, row in df.iterrows():
            cve_ent = row["cve_ent"]
            state = state_master.get(cve_ent)
            if not state:
                continue

            value = row[spec["variable"]]
            if pd.isna(value):
                continue

            long_records.append({
                "state_code": state["state_code"],
                "cve_ent": cve_ent,
                "estado": state["estado"],
                "categoria": spec["categoria"],
                "variable": spec["variable"],
                "valor": round(float(value), 4),
                "anio": spec["anio"],
                "fuente": SOURCE,
                "unidad": spec["unidad"],
            })

    wide_records_by_state = {}
    for record in long_records:
        current = wide_records_by_state.setdefault(record["cve_ent"], {
            "state_code": record["state_code"],
            "cve_ent": record["cve_ent"],
            "estado": record["estado"],
            "region": state_master[record["cve_ent"]]["region"],
            "metrics": {},
        })
        current["metrics"][record["variable"]] = record["valor"]

    wide_records = list(sorted(wide_records_by_state.values(), key=lambda item: item["cve_ent"]))

    long_payload = {
        "updated_at": UPDATED_AT,
        "schema_version": "1.0.0",
        "dataset_id": "context_variables_state_observations.long",
        "source": SOURCE,
        "records": long_records,
    }

    wide_payload = {
        "updated_at": UPDATED_AT,
        "schema_version": "1.0.0",
        "dataset_id": "context_variables_state_dashboard.wide",
        "source": SOURCE,
        "metric_catalog": metric_catalog,
        "records": wide_records,
    }

    return long_payload, wide_payload


def save_outputs(long_payload, wide_payload):
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    PUBLIC_DIR.mkdir(parents=True, exist_ok=True)

    long_json_path = PROCESSED_DIR / "context_variables_state_observations.long.json"
    wide_json_path = PROCESSED_DIR / "context_variables_state_dashboard.wide.json"

    long_json_path.write_text(json.dumps(long_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    wide_json_path.write_text(json.dumps(wide_payload, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Outputs saved:\n  {long_json_path}\n  {wide_json_path}")


long_payload, wide_payload = build_outputs(df_base)
save_outputs(long_payload, wide_payload)


Outputs saved:
  ..\data\processed\context_variables_state_observations.long.json
  ..\data\processed\context_variables_state_dashboard.wide.json
